# Deep Learning Project – Flower classification with CNNs and Transfer Learning

**Team :** Member 1, Member 2, Member 3


**Goal :** classify photos of flowers into 5 categories (daisy, dandelion, roses, sunflowers, tulips).

We compare :

* a simple CNN trained from scratch (baseline)
* an improved CNN (data augmentation, batch normalization, dropout)
* transfer learning with **MobileNetV2** pretrained on ImageNet : feature extraction, then fine-tuning
* an extra experiment : a second pretrained backbone (**EfficientNetB0**)

At the end we look at the errors (confusion matrix) and we use **Grad-CAM** to see where the model is looking.


> Run it on Google Colab with a GPU (Runtime > Change runtime type > GPU), the whole notebook takes around 15 min.

## 0. Setup

In [ ]:
import sys, platform
print(sys.version)
print(platform.system(), platform.machine())

In [ ]:
import os
import pathlib

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import tensorflow as tf
import keras
from keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay


print("TensorFlow :", tf.__version__)
print("Keras :", keras.__version__)
print("GPU :", tf.config.list_physical_devices("GPU"))

In [ ]:
# same seed everywhere so the runs are (almost) reproducible
SEED = 42
keras.utils.set_random_seed(SEED)

IMG_SIZE   = 160
BATCH_SIZE = 32

# number of epochs for each experiment (early stopping will often stop before)
EPOCHS_SCRATCH  = 30
EPOCHS_IMPROVED = 80
EPOCHS_TL       = 15
EPOCHS_FINETUNE = 20

## 1. Dataset

We use the **TensorFlow Flowers** dataset : 3,670 photos of flowers, split in 5 folders (one folder per class).

The photos come from Flickr, they have different sizes, and the background, the light and the framing change a lot from one image to another. It's a small dataset for a CNN, so it's a good case to show the interest of transfer learning.

In [ ]:
URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"

archive = keras.utils.get_file("flower_photos", origin=URL, untar=True)
data_dir = pathlib.Path(archive)

# depending on the keras version the images can be one folder deeper
if not (data_dir / "daisy").exists():
    data_dir = data_dir / "flower_photos"

print(data_dir)

In [ ]:
class_names = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
num_classes = len(class_names)

print("Classes :", class_names)


paths, labels = [], []

for i, name in enumerate(class_names):
    files = sorted((data_dir / name).glob("*.jpg"))
    paths  += [str(f) for f in files]
    labels += [i] * len(files)

paths  = np.array(paths)
labels = np.array(labels)

print("Total images :", len(paths))

In [ ]:
# number of images per class
counts = pd.Series(labels).value_counts().sort_index()
counts.index = class_names

counts.plot(kind="bar", figsize=(7, 4), title="Images per class")
plt.ylabel("count")
plt.xticks(rotation=0)
plt.show()

print(counts)

The classes are not perfectly balanced (dandelion has the most images, roses the fewest) but the difference is small, so accuracy stays a meaningful metric. We will still look at the per-class results at the end.

### Train / validation / test split

70 % train, 15 % validation, 15 % test.

The split is **stratified** so each class keeps the same proportion in the 3 sets. The test set is only used at the very end, to compare the final models. Everything else (early stopping, choices of hyperparameters) is done with the validation set.

In [ ]:
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=SEED
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, stratify=temp_labels, random_state=SEED
)


print("Train :", len(train_paths))
print("Val   :", len(val_paths))
print("Test  :", len(test_paths))

### tf.data pipeline

Each image is read, decoded and resized to 160x160. We keep the pixels in [0, 255] here, the scaling is done **inside** each model (because the pretrained networks expect their own preprocessing).

`cache()` keeps the decoded images in memory after the first epoch (stored as uint8 to save RAM), and `prefetch()` prepares the next batch while the GPU is working on the current one.

In [ ]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.uint8)
    return img, label


def make_dataset(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.cache()

    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED)

    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y))
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(train_paths, train_labels, shuffle=True)
val_ds   = make_dataset(val_paths, val_labels)
test_ds  = make_dataset(test_paths, test_labels)   # not shuffled -> same order as test_labels

In [ ]:
# a few examples from the training set
images, lbls = next(iter(train_ds))

plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[lbls[i]])
    plt.axis("off")
plt.show()

print("Batch shape :", images.shape)

## 2. Helper functions

Small functions we reuse for every model : training, plotting the curves, and storing the test results.

Two callbacks are used for every training :

* **EarlyStopping** : stops when the validation loss doesn't improve anymore and restores the best weights
* **ReduceLROnPlateau** : divides the learning rate by 2 when the validation loss stagnates for a few epochs. Smaller steps at the end of the training make the curves less noisy and help the model to settle in a minimum.

The patience of ReduceLROnPlateau is smaller than the one of EarlyStopping, so the learning rate has time to be reduced before the training is stopped.

In [ ]:
results = {}     # name -> test accuracy
histories = {}   # name -> keras history


def train(model, name, epochs, patience=5, initial_epoch=0):
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True,
        verbose=1
    )

    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=max(2, patience // 2),
        min_lr=1e-7,
        verbose=1
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        initial_epoch=initial_epoch,
        callbacks=[early_stop, reduce_lr]
    )
    histories[name] = history
    return history


def plot_history(history, title):
    h = history.history
    epochs = range(1, len(h["loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(epochs, h["accuracy"], "o-", label="train")
    axes[0].plot(epochs, h["val_accuracy"], "s-", label="validation")
    axes[0].set_title(title + " - accuracy")

    axes[1].plot(epochs, h["loss"], "o-", label="train")
    axes[1].plot(epochs, h["val_loss"], "s-", label="validation")
    axes[1].set_title(title + " - loss")

    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.grid(alpha=0.3)
        ax.legend()

    plt.tight_layout()
    plt.show()


def evaluate(model, name):
    loss, acc = model.evaluate(test_ds, verbose=0)
    results[name] = acc
    print(f"{name}  ->  test loss = {loss:.4f}   test accuracy = {acc:.2%}")

## 3. Baseline : CNN from scratch

A classic architecture : 3 blocks **Conv2D + ReLU + MaxPooling**, with 32, 64 then 128 filters, then a dense classifier.

* the convolutions learn local patterns (edges, colors, textures, then petal shapes)
* max pooling divides the spatial size by 2 each time, so the deeper filters see a bigger part of the image
* the last layer is a softmax over the 5 classes

The labels are integers (0 to 4), so we use `sparse_categorical_crossentropy`.

In [ ]:
def build_baseline():
    model = keras.Sequential([
        keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.Rescaling(1. / 255),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ], name="baseline_cnn")

    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


baseline = build_baseline()
baseline.summary()

Most of the parameters are in the first Dense layer : the Flatten gives 20 x 20 x 128 = 51,200 values, connected to 128 neurons. That's a lot of parameters for only ~2,500 training images, so we expect overfitting.

In [ ]:
history_baseline = train(baseline, "Baseline CNN", EPOCHS_SCRATCH, patience=5)

In [ ]:
plot_history(history_baseline, "Baseline CNN")
evaluate(baseline, "Baseline CNN")

**Observation.** The training accuracy keeps going up quickly (close to 100 % after 8 epochs) while the validation accuracy stays stuck around 65-67 %, and the validation loss goes back up after epoch 3 : the model **overfits**, it memorizes the training images instead of learning general features. Early stopping gives us back the weights of the best epoch, but the gap between train and validation is huge.

## 4. Improved CNN

Same idea but with the regularization techniques we saw in Lab 3 :

* **data augmentation** : random flips, rotations, zooms and contrast changes. These layers are only active during training. A flower stays the same flower if the photo is flipped or slightly rotated, so it creates "new" training examples for free.
* **batch normalization** after each convolution : stabilizes training and allows a faster convergence
* **GlobalAveragePooling** instead of Flatten : each feature map becomes one number, so much fewer parameters in the classifier
* **dropout (0.5)** before the last layer

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name="data_augmentation")


# what one image looks like after augmentation
img = images[0:1]

plt.figure(figsize=(10, 10))
for i in range(9):
    aug = data_augmentation(img, training=True)
    plt.subplot(3, 3, i + 1)
    plt.imshow(np.clip(aug[0].numpy(), 0, 255).astype("uint8"))
    plt.axis("off")
plt.show()

In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.MaxPooling2D()(x)
    return x


def build_improved():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1. / 255)(x)

    for filters in [32, 64, 128, 256]:
        x = conv_block(x, filters)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="improved_cnn")
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


improved = build_improved()
improved.summary()

The convolutions use `use_bias=False` because the BatchNormalization just after already has its own shift parameter (beta), a bias would be useless.

In [ ]:
history_improved = train(improved, "Improved CNN", EPOCHS_IMPROVED, patience=10)

In [ ]:
plot_history(history_improved, "Improved CNN")
evaluate(improved, "Improved CNN")

**Observation.** The train and validation curves stay much closer : the augmentation makes the training task harder, so the model can't simply memorize. The test accuracy is clearly better than the baseline.

In our first run (fixed learning rate, 50 epochs) the validation accuracy was jumping a lot from one epoch to the next (between ~67 % and ~82 %) and the best epoch was the 47th, so the model was still learning when the training stopped. That's why we added ReduceLROnPlateau and more epochs : once the learning rate is reduced, the validation curve becomes much smoother.

Still, with ~2,500 images it's hard to learn good visual features from zero. That's where transfer learning helps.

## 5. Transfer learning : MobileNetV2

**Idea :** a network trained on ImageNet (1.2 million images, 1000 classes) has already learned generic visual features : edges and colors in the first layers, textures and object parts in the deeper ones. We reuse these features and only train a new classifier for our 5 classes.

We choose **MobileNetV2** : light (~2.3 M parameters in the base), fast, and good on ImageNet. It uses depthwise separable convolutions and inverted residual blocks.

Two steps :

1. **Feature extraction** : the base is frozen, we only train the new head
2. **Fine-tuning** : we unfreeze the last layers of the base and train them with a very small learning rate

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,          # we remove the ImageNet classifier
    weights="imagenet"
)

base_model.trainable = False   # freeze all the convolutional layers

print("Layers in the base model :", len(base_model.layers))
print("Output shape :", base_model.output_shape)

MobileNetV2 expects pixels in [-1, 1], so we use its own `preprocess_input` function.

We call the base with `training=False` : this keeps the BatchNormalization layers in inference mode (they use the ImageNet statistics). It's important for the fine-tuning later, otherwise the batch norm statistics would be updated with our small batches and destroy what the network learned.

In [ ]:
preprocess = keras.applications.mobilenet_v2.preprocess_input


def build_transfer(base, preprocess_fn, name):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = preprocess_fn(x)
    x = base(x, training=False)

    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.2, name="head_dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    model = keras.Model(inputs, outputs, name=name)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


tl_model = build_transfer(base_model, preprocess, "mobilenetv2_tl")
tl_model.summary()

Only the last Dense layer is trainable : 1280 x 5 + 5 = 6,405 parameters, compared to millions in the frozen base.

In [ ]:
history_tl = train(tl_model, "MobileNetV2 - feature extraction", EPOCHS_TL, patience=4)

In [ ]:
plot_history(history_tl, "MobileNetV2 - feature extraction")
evaluate(tl_model, "MobileNetV2 - feature extraction")

**Observation.** Already after the first epoch the validation accuracy is above what the CNNs trained from scratch reach at the end. The pretrained features are very well suited to natural images like flowers.

(The validation accuracy can even be above the train accuracy here : dropout and data augmentation are only active during training, which makes the training task harder.)

### Fine-tuning

Now we unfreeze the **top layers** of MobileNetV2 (from layer 100 to the end). The first layers detect very generic things (edges, colors) so we keep them frozen, the last ones are more specific to ImageNet objects, so it makes sense to adapt them to flowers.

The learning rate is divided by 100 (1e-5) : the weights are already good, we only want to adjust them a little. A big learning rate would destroy the pretrained features at the start of the fine-tuning, when the gradients are still large.

We have to **compile again** after changing `trainable`, otherwise the change is not taken into account.

In [ ]:
base_model.trainable = True

fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False


n_trainable = sum(l.trainable for l in base_model.layers)
print(f"Trainable layers in the base : {n_trainable} / {len(base_model.layers)}")


tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
start = len(history_tl.history["loss"])

history_ft = train(
    tl_model,
    "MobileNetV2 - fine-tuning",
    epochs=start + EPOCHS_FINETUNE,
    initial_epoch=start,
    patience=5
)

In [ ]:
# full curve : feature extraction + fine-tuning
acc      = history_tl.history["accuracy"] + history_ft.history["accuracy"]
val_acc  = history_tl.history["val_accuracy"] + history_ft.history["val_accuracy"]
loss     = history_tl.history["loss"] + history_ft.history["loss"]
val_loss = history_tl.history["val_loss"] + history_ft.history["val_loss"]

epochs = range(1, len(acc) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs, acc, "o-", label="train")
axes[0].plot(epochs, val_acc, "s-", label="validation")
axes[0].set_title("MobileNetV2 - accuracy")

axes[1].plot(epochs, loss, "o-", label="train")
axes[1].plot(epochs, val_loss, "s-", label="validation")
axes[1].set_title("MobileNetV2 - loss")

for ax in axes:
    ax.axvline(start + 0.5, color="gray", linestyle="--", label="start fine-tuning")
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()


evaluate(tl_model, "MobileNetV2 - fine-tuning")

**Observation.** Fine-tuning gives an extra gain on top of feature extraction, but a smaller one : most of the work was already done by the frozen features.

Right after unfreezing, the training accuracy drops a bit for one or two epochs (the optimizer is restarted and the unfrozen layers begin to move), while the validation accuracy doesn't : the pretrained features are not destroyed thanks to the small learning rate. We have to watch the gap between train and validation, because the model now has ~1.9 M trainable parameters and can overfit again.

## 6. Extra experiment : another backbone (EfficientNetB0)

Is MobileNetV2 a good choice ? We try **EfficientNetB0** with exactly the same protocol (frozen base + new head, then fine-tuning) so the comparison is fair.

EfficientNet scales the depth, the width and the input resolution of the network together. It is more accurate on ImageNet than MobileNetV2 but also a bit heavier (~4 M parameters in the base).

Note : in Keras, EfficientNet includes its own rescaling layer, so it takes the pixels directly in [0, 255], the `preprocess_input` function does nothing.

In [ ]:
effnet_base = keras.applications.EfficientNetB0(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)
effnet_base.trainable = False


effnet_model = build_transfer(
    effnet_base,
    keras.applications.efficientnet.preprocess_input,
    "efficientnetb0_tl"
)

history_effnet = train(effnet_model, "EfficientNetB0 - feature extraction", EPOCHS_TL, patience=4)

In [ ]:
plot_history(history_effnet, "EfficientNetB0 - feature extraction")
evaluate(effnet_model, "EfficientNetB0 - feature extraction")

### Fine-tuning EfficientNetB0

Same idea as for MobileNetV2 : we unfreeze the last layers of the base (the last 40 layers, about the last 2 blocks) and train with a learning rate of 1e-5.

As recommended in the Keras transfer learning guide, the **BatchNormalization layers stay frozen**. They already run in inference mode because of `training=False`, and freezing them also keeps their scale and shift parameters (gamma, beta) as they were learned on ImageNet.

In [ ]:
effnet_base.trainable = True

for layer in effnet_base.layers[:-40]:
    layer.trainable = False

# keep the batch norm layers frozen
for layer in effnet_base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False


n_trainable = sum(l.trainable for l in effnet_base.layers)
print(f"Trainable layers in the base : {n_trainable} / {len(effnet_base.layers)}")


effnet_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

start_eff = len(history_effnet.history["loss"])

history_effnet_ft = train(
    effnet_model,
    "EfficientNetB0 - fine-tuning",
    epochs=start_eff + EPOCHS_FINETUNE,
    initial_epoch=start_eff,
    patience=5
)

In [ ]:
plot_history(history_effnet_ft, "EfficientNetB0 - fine-tuning")
evaluate(effnet_model, "EfficientNetB0 - fine-tuning")

In [ ]:
# parameters of each model (total / trainable during the last training phase)
models_list = {
    "Baseline CNN": baseline,
    "Improved CNN": improved,
    "MobileNetV2 (fine-tuned)": tl_model,
    "EfficientNetB0 (fine-tuned)": effnet_model,
}

rows = []
for name, m in models_list.items():
    trainable = int(sum(np.prod(w.shape) for w in m.trainable_weights))
    rows.append({"Model": name, "Total params": m.count_params(), "Trainable params": trainable})

pd.DataFrame(rows)

## 7. Comparison of all the models

The test set has 551 images, so a test accuracy is only an estimate. We add a **95 % confidence interval** with the normal approximation : `± 1.96 * sqrt(p (1 - p) / n)`.

For an accuracy around 90 % this gives about ± 2.5 points : two models whose intervals overlap a lot can't really be separated with this test set.

In [ ]:
n_test = len(test_labels)

summary = (
    pd.DataFrame({"Model": list(results.keys()), "Test accuracy": list(results.values())})
    .sort_values("Test accuracy", ascending=False)
    .reset_index(drop=True)
)

summary["95% CI (+/-)"] = 1.96 * np.sqrt(summary["Test accuracy"] * (1 - summary["Test accuracy"]) / n_test)

summary.index = summary.index + 1
print(summary.to_string(formatters={"Test accuracy": "{:.2%}".format, "95% CI (+/-)": "{:.2%}".format}))


plt.figure(figsize=(9, 4))
plt.barh(summary["Model"][::-1], summary["Test accuracy"][::-1], xerr=summary["95% CI (+/-)"][::-1], capsize=4)
plt.xlabel("Test accuracy")
plt.xlim(0, 1)
plt.grid(alpha=0.3, axis="x")
plt.title("Test accuracy per model (with 95 % confidence interval)")
plt.show()

In [ ]:
# validation accuracy of every experiment on the same plot
plt.figure(figsize=(10, 6))

for name in ["Baseline CNN", "Improved CNN", "MobileNetV2 - feature extraction", "EfficientNetB0 - feature extraction"]:
    val = histories[name].history["val_accuracy"]
    plt.plot(range(1, len(val) + 1), val, marker="o", markersize=3, label=name)

plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("Validation accuracy during training")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 8. Error analysis

We now look in more detail at the fine-tuned MobileNetV2 : which classes are confused with each other ?

We keep MobileNetV2 here (and not EfficientNet) because the two transfer learning models are within the confidence interval of each other, and MobileNetV2 is lighter and faster, which would be the better choice for a real application (e.g. on a phone).

In [ ]:
best_model = tl_model

probs  = best_model.predict(test_ds, verbose=0)
y_pred = probs.argmax(axis=1)
y_true = test_labels


print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion matrix - MobileNetV2 fine-tuned (test set)")
plt.show()

In [ ]:
# some of the misclassified images
wrong = np.where(y_pred != y_true)[0]
print("Misclassified :", len(wrong), "/", len(y_true))


plt.figure(figsize=(14, 7))

for i, idx in enumerate(wrong[:10]):
    img, _ = load_image(test_paths[idx], 0)

    plt.subplot(2, 5, i + 1)
    plt.imshow(img.numpy())
    plt.title(f"true: {class_names[y_true[idx]]}\npred: {class_names[y_pred[idx]]} ({probs[idx].max():.0%})", fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.show()

**Observation.** The most frequent error is **roses predicted as tulips** (16 test images in our run) : both are often red or pink with similar petals, and in many photos only the flower head is visible. Roses also have the lowest recall.

Looking at the misclassified images, a part of the errors is not really the fault of the model :

* some labels are questionable (for example a "tulip" image that actually shows a daffodil in a field of hyacinths)
* some photos are very hard : heavy color filters, a flower that is tiny in the picture, a building that takes most of the image

So the dataset itself is a bit noisy, and part of the remaining ~10 % of errors can't be fixed by a better model.

## 9. Grad-CAM : where does the model look ?

Grad-CAM (Selvaraju et al., 2017) produces a heatmap of the regions of the image that were important for the prediction.

How it works :

1. we take the feature maps of the last convolutional layer (5 x 5 x 1280 for MobileNetV2 at 160x160)
2. we compute the gradient of the score of the predicted class with respect to these feature maps
3. we average the gradient over each map : this gives one weight per feature map (how much it matters for this class)
4. weighted sum of the maps, then ReLU (we only keep the positive influence), then we resize it to the size of the image

In [ ]:
def grad_cam(img):
    x = tf.convert_to_tensor(img[None, ...], dtype=tf.float32)
    x = preprocess(x)

    with tf.GradientTape() as tape:
        feature_maps = base_model(x, training=False)       # (1, 5, 5, 1280)
        tape.watch(feature_maps)

        pooled = tl_model.get_layer("gap")(feature_maps)
        preds  = tl_model.get_layer("predictions")(pooled)
        class_idx = tf.argmax(preds[0])
        score = preds[:, class_idx]

    grads = tape.gradient(score, feature_maps)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))         # one weight per feature map

    cam = tf.reduce_sum(feature_maps[0] * weights, axis=-1)
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam) + 1e-8)

    cam = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE))[..., 0]
    return cam.numpy(), int(class_idx)

In [ ]:
def show_grad_cam(indices, title):
    plt.figure(figsize=(15, 6))

    for k, idx in enumerate(indices):
        img, _ = load_image(test_paths[idx], 0)
        img = img.numpy()

        heatmap, pred = grad_cam(img.astype("float32"))

        plt.subplot(2, len(indices), k + 1)
        plt.imshow(img)
        plt.title(f"true: {class_names[y_true[idx]]}")
        plt.axis("off")

        plt.subplot(2, len(indices), k + 1 + len(indices))
        plt.imshow(img)
        plt.imshow(heatmap, cmap="jet", alpha=0.4)
        plt.title(f"pred: {class_names[pred]}")
        plt.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


# one correctly classified image per class
correct = [np.where((y_true == i) & (y_pred == i))[0][0] for i in range(num_classes)
           if np.any((y_true == i) & (y_pred == i))]
show_grad_cam(correct, "Correctly classified")

# and 5 errors
show_grad_cam(wrong[:5], "Misclassified")

**Observation.** On the correctly classified images the heatmap is mostly on the flowers (petals, heart of the flower) and not on the background, so the model uses the right information.

On the errors it's different : when the flower is small or when the image contains a lot of other things (a field, leaves, a building), the heatmap is spread over the whole scene or focused on the background. For example, in our first run, a photo of a sunflower field was predicted as daisy with the heatmap in the middle of the field and not on the sunflowers. The model then relies on the context instead of the flower.

The map is quite coarse because the last feature maps are only 5 x 5.

## 10. Conclusion

* A CNN trained **from scratch** on ~2,500 images overfits very quickly (~66 % test accuracy). Data augmentation, batch normalization and dropout reduce the overfitting and bring it above 80 %, but the model stays limited by the small amount of data.
* **Transfer learning** gives the biggest improvement : with a frozen MobileNetV2 and only 6,405 trainable parameters we get close to 90 % in a few epochs. The features learned on ImageNet transfer very well to flowers.
* **Fine-tuning** the top layers with a small learning rate adds a little more.
* MobileNetV2 and EfficientNetB0 give results in the same range, and the difference between them is smaller than the confidence interval of our test set : on this dataset the choice of the backbone matters much less than the fact of using a pretrained one.
* The remaining errors are mostly roses / tulips confusions, difficult photos and some questionable labels. Grad-CAM shows that the model looks at the flowers when it is right, and often at the background when it is wrong.

**Possible improvements :** a bigger input resolution (224x224), unfreezing more layers, cleaning the mislabeled images, a bigger test set (or k-fold cross-validation) to compare the models more reliably, or test-time augmentation.